In [2]:
import sys
sys.path.insert(0, '..')


# Basic imports
import jax.numpy as np
import jax.random as jr
import jax.scipy as jsp
import jax
import numpy as onp
import pickle

jax.config.update("jax_enable_x64", True)


# Optimisation imports
import zodiax as zdx
import optax
import optimistix as optx
import equinox as eqx

# dLux imports
import dLux as dl
import dLux.utils as dlu

#FITS 
from astropy.io import fits
from pathlib import Path
import pandas as pd

# Visualisation imports
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
import chainconsumer as cc


%matplotlib inline
plt.rcParams['image.cmap'] = 'inferno'
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = 'lower'
plt.rcParams['figure.dpi'] = 72
plt.rcParams["font.size"] = 24

from detectors import *
from apertures import *
from models import *
from stats import posterior
from fitting import *
from plotting import *
from spectra import *

import jax.tree_util as jtu
import interpax as ipx

def set_array(pytree):
    dtype = np.float64 if jax.config.x64_enabled else np.float32
    floats, other = eqx.partition(pytree, eqx.is_inexact_array_like)
    floats = jtu.tree_map(lambda x: np.array(x, dtype=dtype), floats)
    return eqx.combine(floats, other)

def to_numpy_tree(tree):
    return jtu.tree_map(
        lambda x: onp.asarray(x)
        if hasattr(x, "shape")
        else x,
        tree
    )

In [6]:
# files = [
#     "../FOSE7902/data/MAST_10143/MAST/HST/n8yj19l6q_cal.fits",
#     # add the other FITS files here
# ]


In [7]:

# # creates an empty list
# file_info = []


# for fname in files:

#     # open the fits file
#     with fits.open(fname) as hdul:
#         filt = hdul[0].header["FILTER"]

#     file_info.append({"filename":fname,"filter": filt})

#     # only take filename at the end 
#     print(Path(fname).name, "->", filt)

# # get the filter in the dictionary , remove duplicate and sort them
# filters = sorted (set(info["filter"] for info in file_info))
    
# print("\nFilters found:")
# print(filters)

In [8]:
# # creating a list of exposures 
# exposures_single = []
# exposures_binary = []

# for info in file_info:

#     fname = info["filename"]
#     filt = info["filter"]

#     exp_single = exposure_from_file(fname,SinglePointFit(spectrum_basis,filt), crop = wid)

#     exp_binary = exposure_from_file(fname,BinaryFit(spectrum_basis,filt),crop=wid)

#     exposures_single.append(exp_single)

#     exposures_binary.append(exp_binary)

In [9]:
# single_exposures_by_filter = {}

# for filt in filters:
#     matching_exposures =[]

#     for exp in exposures_single:
#         if exp.filter == filt:
#             matching_exposures.append(exp)

#     single_exposures_by_filter[filt] = matching_exposures


# binary_exposures_by_filter ={}

# for filt in filters:
#     matching_exposures = []

#     for exp in exposures_binary:
#         if exp.filter == filt:
#             matching_exposures.append(exp)
    
#     binary_exposures_by_filter[filt] = matching_exposures 

In [10]:

# # Base NICMOS parameters

# params = {
#     "positions": {},
#     "spectrum": {},
#     "aberrations": {},
#     "cold_mask_shift": {},
#     "cold_mask_rot": {},
#     "cold_mask_scale": {},
#     "cold_mask_shear": {},
#     "primary_scale": {},
#     "primary_rot": {},
#     "primary_shear": {},
#     "bias": {},

#     "outer_radius": 1.2 * 0.955,
#     "secondary_radius": 0.372 * 1.2,
#     "spider_width": 0.077 * 1.2,

#     "scale": 0.0432,
#     "rot": 0.0,
#     "softening": 2.0,
# }

In [11]:
# grid_results = {}

In [12]:
# for filt in filters:

#     print("\n")
#     print("=" * 60)
#     print("FILTER:", filt)
#     print("=" * 60)

#     exposures_single_filt = (
#         single_exposures_by_filter[filt]
#     )

#     exposures_binary_filt = (
#         binary_exposures_by_filter[filt]
#     )



In [13]:

# for exp in exposures_single_filt:
#     params["positions"][exp.fit.get_key(exp, "positions")] = np.asarray([0.,0.])
#     params["spectrum"][exp.fit.get_key(exp, "spectrum")] = np.zeros(npoly).at[0].set(1)*np.log10(np.nansum(exp.data)/nwavels)
#     params["aberrations"][exp.fit.get_key(exp, "aberrations")] = np.zeros(26)
#     params["cold_mask_shift"][exp.fit.get_key(exp, "cold_mask_shift")] = -np.asarray([-0.06, -0.06])*1e2
#     params["cold_mask_rot"][exp.fit.get_key(exp, "cold_mask_rot")] = -45.
#     params["cold_mask_scale"][exp.fit.get_key(exp, "cold_mask_scale")] = np.asarray([1.,1.])
#     params["cold_mask_shear"][exp.fit.get_key(exp, "cold_mask_shear")] = np.asarray([0.,0.])
#     params["primary_rot"][exp.fit.get_key(exp, "primary_rot")] = -45. #+ 180.
#     params["primary_scale"][exp.fit.get_key(exp, "primary_scale")] = np.asarray([1.,1.])
#     params["primary_shear"][exp.fit.get_key(exp, "primary_shear")] = np.asarray([0.,0.])

#     params["bias"][exp.fit.get_key(exp, "bias")] = 0.

In [14]:
# # single source model at normal detector resolution
# model_normal = set_array(NICMOSModel(exposures_single_filt,params,optics,detector))

# # Same single source model but with much finer PSF sampling 
# # This is what we use for the FFT shifting 
# model_oversampled=set_array(NICMOSModel(exposures_single_filt,params,oversampled_optics,detector))

# #Proper Binary fit version used later for the grid search
# model_binary = set_array(NICMOSModel(exposures_binary_filt,params, optics,detector))

# params = ModelParams(params)

In [15]:
# # generate oversampled PSF
# # Basically this is the template we about the slide around until it lines up with the actual NICMOS observation
# psf = exposures_single_filt[0].fit(model_oversampled,exposures_single_filt[0])


In [ ]:
# FFT-based sub-pixel image shifting
fft = np.fft.fft2
ifft = np.fft.ifft2
fftshift = np.fft.fftshift


def reposition(image, dx, dy):
    """
    Shift an image by dx, dy pixels using FFT.
    """

    # Image dimensions
    Nx, Ny = image.shape

    # Frequency grid
    kx = fftshift(np.fft.fftfreq(Nx, d=1.0))
    ky = fftshift(np.fft.fftfreq(Ny, d=1.0))

    # Fourier phase ramp
    phase_ramp = np.exp(-2.j * np.pi *(kx[:, None] * dx + ky[None, :] * dy))

    # Apply shift in Fourier space
    shifted_image = fftshift(ifft(fft(fftshift(image))* fftshift(phase_ramp) ) )

    return np.abs(shifted_image)

In [ ]:
def fit_flux(exp, template):

    psf = template.flatten()

    bad = exp.bad.flatten()
    err = exp.err.flatten()

    psf= np.where(exp.bad.flatten(), 0., psf)

    psf = psf/np.sum(psf)

    data = exp.data.flatten()
        #data= data.at[exp.bad.flatten()].set(0.)
    data= np.where(exp.bad.flatten(), 0., data)


    design = np.transpose(np.vstack((np.ones(len(psf)), psf)))

    flux_raw, _, _, _ = np.linalg.lstsq(design, data)
    true_flux = flux_raw[1]        
    #true_flux= np.log10(flux_raw[1]/nwavels * 2/(1+contrast)) - spec
        #print(true_flux)
    
    return -np.sum(np.where(bad, 0., jsp.stats.norm.logpdf(psf*true_flux, data, err))), true_flux#loss(template*true_flux, exp)

In [18]:
def flux_to_spectrum(flux, filt,spectrum_basis, nwavels, npoly):

    # Filter throughput sampled at the same wavelengths as the model
    _, throughput = calc_throughput(filt,nwavels=nwavels)

    # Start with all polynomial coefficients = 0
    spectrum = np.zeros(npoly)

    # Flux produced by coefficient 0 = 1
    unit_flux = np.sum(throughput * spectrum_basis[:, 0])

    # Set only the first coefficient
    spectrum = spectrum.at[0].set(flux / unit_flux)

    return spectrum

In [ ]:
@eqx.filter_jit
def grid_loss_single(dx,dy,psf,exposure,psf_oversample,return_flux=False,return_psf=False):

    psf_shifted = dlu.downsample(reposition(psf,dx * psf_oversample,dy * psf_oversample),psf_oversample)

    loss, flux = fit_flux(exposure,psf_shifted)

    if return_psf:
        return psf_shifted

    if return_flux:
        return loss, flux

    return loss

In [ ]:
def run_single_grid_search(psf,exposure,psf_oversample,x_min=-5,x_max=5,y_min=-5,y_max=5,Nx=50,Ny=50,verbose=True,):

       xs = np.linspace(x_min, x_max, Nx)
       ys = np.linspace(y_min, y_max, Ny)

       # a 50 by 50 zero array
       single_losses = np.zeros((Nx, Ny))
       
       for i, dx in enumerate(tqdm(xs,desc ="Single grid")):
                for j,dy in enumerate(ys):
                        single_losses = single_losses.at[i,j].set(grid_loss_single(dx,dy,psf,exposure,psf_oversample))

        # Find the best single solution
        # takes the flattened index and converts back to 2D
       xi, yi  = np.unravel_index(single_losses.argmin(),single_losses.shape)
       best_x = xs[xi]
       best_y = ys[yi]

       single_loss, single_flux = (grid_loss_single(best_x,best_y,psf,exposure,psf_oversample, return_flux = True))
       
       # Store the results
       result = {
               "best_x": best_x,
               "best_y": best_y,
               "loss": single_loss,
               "flux": single_flux,
               "loss_grid": single_losses
        }
       
       if verbose:
                print("Best x shift: ", best_x)
                print("Best y shift: ", best_y)
                print("Single loss : ", single_loss)
                print("Single flux : ", single_flux)

       return result
                
        


In [ ]:
def extract_single_params(single_result, exposures, spectrum_basis, nwavels, npoly):

    single_params = {"positions":{},"spectrum":{}}

    for exp in exposures:
        
      # Position key is exposure-specific
        position_key = exp.fit.get_key( exp, "positions" )

        single_params["positions"][position_key] = np.asarray([single_result["best_x"],single_result["best_y"]])

        # Spectrum is shared by target + filter
        spectrum_key = exp.fit.get_key(exp,"spectrum")

        single_params["spectrum"][spectrum_key] = flux_to_spectrum(single_result["flux"],exp.filter, spectrum_basis, nwavels, npoly)

    return ModelParams(single_params)
   

In [ ]:
@eqx.filter_jit
def grid_loss_binary(best_x,best_y,r,theta,contrast,psf,exposure,psf_oversample,return_flux=False,return_psf=False):

    dx1 = best_x
    dy1 = best_y

    ang = dlu.deg2rad(theta)

    dx2 = dx1 - r * np.cos(ang)
    dy2 = dy1 - r * np.sin(ang)

    primary_shifted = reposition(psf,dx1 * psf_oversample,dy1 * psf_oversample)
    secondary_shifted = reposition(psf,dx2 * psf_oversample,dy2 * psf_oversample)

    f1, f2 = dlu.fluxes_from_contrast(1.0,contrast)

    psf_total = dlu.downsample(f1 * primary_shifted + f2 * secondary_shifted, psf_oversample)

    loss, flux = fit_flux(exposure,psf_total)

    if return_psf:
        return psf_total

    if return_flux:
        return loss, flux

    return loss

In [ ]:
def run_binary_grid_search(best_x,best_y, psf,exposure,psf_oversample,r_min=0,r_max=5.0,Nr=20,theta_min=-180,theta_max=180,Ntheta=50,c_min=1.0,c_max=10.0,Nc=20,verbose=True):
    
    rs = np.linspace(r_min,r_max, Nr)
    thetas = np.linspace(theta_min, theta_max, Ntheta)
    contrasts = 10**(np.linspace(np.log10(c_min), np.log10(c_max), Nc))


    best_binary_loss = np.inf
    best_r = None
    best_theta = None
    best_contrast = None
    best_binary_flux = None

    for r in tqdm(rs, desc = "Binary separation"):
        for theta in thetas:
            for contrast in contrasts:
                loss, flux = grid_loss_binary(best_x,best_y,r,theta,contrast,psf,exposure,psf_oversample, return_flux =True)

                if loss< best_binary_loss:
                    best_binary_loss = loss
                    best_r = r
                    best_theta = theta
                    best_contrast = contrast
                    best_binary_flux = flux

    result = {
        "best_x": best_x,
        "best_y": best_y,
        "best_r": best_r,
        "best_theta": best_theta,
        "best_contrast": best_contrast,
        "loss": best_binary_loss,
        "flux": best_binary_flux,
    }

    if verbose:
        print("Best x shift:", best_x)
        print("Best y shift:", best_y)
        print("Best separation:", best_r)
        print("Best position angle:", best_theta)
        print("Best contrast:", best_contrast)
        print("Binary loss:", best_binary_loss)
        print("Binary flux:", best_binary_flux)

    return result



In [ ]:
def calculate_binary_components(binary_result):

    # Get best-fit values from the binary grid
    best_x = binary_result["best_x"]
    best_y = binary_result["best_y"]
    best_r = binary_result["best_r"]
    best_theta = binary_result["best_theta"]
    best_contrast = binary_result["best_contrast"]
    total_flux = binary_result["flux"]

    # Convert angle to radians
    ang = dlu.deg2rad(best_theta)

    # Primary position
    primary_x = best_x
    primary_y = best_y

    # Secondary position
    secondary_x = primary_x - best_r * np.cos(ang)
    secondary_y = primary_y - best_r * np.sin(ang)

    # Get relative component fluxes
    f1, f2 = dlu.fluxes_from_contrast( 1.0,best_contrast)

    # Normalise so the two fractions sum to 1
    flux_sum = f1 + f2

    f1 = f1 / flux_sum
    f2 = f2 / flux_sum

    # Split total flux
    primary_flux = total_flux * f1
    secondary_flux = total_flux * f2

    return {
        "primary_x": primary_x,
        "primary_y": primary_y,
        "secondary_x": secondary_x,
        "secondary_y": secondary_y,
        "primary_flux": primary_flux,
        "secondary_flux": secondary_flux,
    }

In [ ]:
def extract_binary_params(binary_result,exposures, spectrum_basis, nwavels, npoly):
    r = binary_result["best_r"]
    theta = binary_result["best_theta"]

    best_x = binary_result["best_x"]
    best_y = binary_result["best_y"]

    ang = dlu.deg2rad(theta)

    # Centre of the binary in the grid
    midpoint_x = best_x - r * np.cos(ang) / 2
    midpoint_y = best_y - r * np.sin(ang) / 2

    binary_params = {"positions": {},"primary_spectrum": {}, "secondary_spectrum": {},"separation": r,"position_angle": theta}

    for exp in exposures:

        position_key = exp.fit.get_key(exp, "positions" )

        # BinaryFit position convention relative to our FFT grid
        binary_params["positions"][position_key] = np.asarray([midpoint_y,midpoint_x])

        primary_key = exp.fit.get_key(exp, "primary_spectrum")

        binary_params["primary_spectrum"][primary_key] = (flux_to_spectrum(binary_result["primary_flux"],exp.filter,spectrum_basis,nwavels,npoly))

        secondary_key = exp.fit.get_key(exp,"secondary_spectrum")

        binary_params["secondary_spectrum"][secondary_key] = (flux_to_spectrum(binary_result["secondary_flux"],exp.filter,spectrum_basis,nwavels,npoly))

    return ModelParams(binary_params)

In [ ]:
def run_grid_for_group( target, filt, exposures_single, exposures_binary, optics, oversampled_optics, detector, spectrum_basis, config, results_dir, make_plots=False, save=True ):

    print()
    print("=" * 60)
    print("TARGET:", target)
    print("FILTER:", filt)
    print("=" * 60)

    # Configuration
    nwavels = config["nwavels"]
    npoly = config["npoly"]
    psf_oversample = ( config["psf_oversample"] )
    n_aberrations = ( config["n_aberrations"] )


    exposure = exposures_single[0]

    
    # Starting parameters
    
    params = {
        "positions": {},
        "spectrum": {},
        "aberrations": {},

        "cold_mask_shift": {},
        "cold_mask_rot": {},
        "cold_mask_scale": {},
        "cold_mask_shear": {},

        "primary_scale": {},
        "primary_rot": {},
        "primary_shear": {},

        "bias": {},
        "outer_radius": 1.2 * 0.955,
        "secondary_radius": 0.372 * 1.2,
        "spider_width": 0.077 * 1.2,
        "scale": 0.0432,
        "rot": 0.0,
        "softening": 2.0,
    }


    for exp in exposures_single:

        params["positions"][ exp.fit.get_key( exp, "positions" ) ] = np.asarray( [0.0, 0.0] )

        params["spectrum"][ exp.fit.get_key( exp, "spectrum" ) ] = ( np.zeros(npoly) .at[0] .set(1) * np.log10( np.nansum(exp.data) / nwavels ) )

        params["aberrations"][ exp.fit.get_key( exp, "aberrations" ) ] = np.zeros( n_aberrations )

        params["cold_mask_shift"][ exp.fit.get_key( exp, "cold_mask_shift" ) ] = ( -np.asarray( [-0.06, -0.06] ) * 1e2 )

        params["cold_mask_rot"][ exp.fit.get_key( exp, "cold_mask_rot" ) ] = -45.0

        params["cold_mask_scale"][ exp.fit.get_key( exp, "cold_mask_scale" ) ] = np.asarray( [1.0, 1.0] )

        params["cold_mask_shear"][ exp.fit.get_key( exp, "cold_mask_shear" ) ] = np.asarray( [0.0, 0.0] )

        params["primary_rot"][ exp.fit.get_key( exp, "primary_rot" ) ] = -45.0

        params["primary_scale"][ exp.fit.get_key( exp, "primary_scale" ) ] = np.asarray( [1.0, 1.0] )

        params["primary_shear"][ exp.fit.get_key( exp, "primary_shear" ) ] = np.asarray( [0.0, 0.0] )

        params["bias"][ exp.fit.get_key( exp, "bias" ) ] = 0.0


    # Oversampled model / PSF

    model_oversampled = set_array( NICMOSModel( exposures_single, params, oversampled_optics, detector ) )
    psf = exposure.fit( model_oversampled, exposure )


    
    # SINGLE GRID
    single_result = ( run_single_grid_search( psf, exposure, psf_oversample, x_min=config["x_min"], x_max=config["x_max"], y_min=config["y_min"], y_max=config["y_max"], Nx=config["Nx"], Ny=config["Ny"], ) )
    single_grid_params = ( extract_single_params( single_result, exposures_single, spectrum_basis, nwavels, npoly ) )
    
    # BINARY GRID
   
    binary_result = ( run_binary_grid_search( single_result["best_x"], single_result["best_y"], psf, exposure, psf_oversample, r_min=config["r_min"], r_max=config["r_max"], Nr=config["Nr"], theta_min= config["theta_min"], theta_max= config["theta_max"], Ntheta= config["Ntheta"], c_min=config["c_min"], c_max=config["c_max"], Nc=config["Nc"], ) )
    binary_components = ( calculate_binary_components( binary_result ) )
    binary_result = { **binary_result, **binary_components }
    binary_grid_params = ( extract_binary_params( binary_result, exposures_binary, spectrum_basis, nwavels, npoly ) )

    
    # Optional plots
   

    if make_plots:

        model_single = set_array( NICMOSModel( exposures_single, single_grid_params.params, optics, detector ) )
        plot_comparison( model_single, single_grid_params, exposures_single )

        model_binary = set_array( NICMOSModel( exposures_binary, binary_grid_params.params, optics, detector ) )
        plot_comparison( model_binary, binary_grid_params, exposures_binary )


   
    # Output

    grid_output = {

        "target": target,
        "filter": filt,
        "single_result": to_numpy_tree( single_result ),
        "binary_result": to_numpy_tree( binary_result ),
        "single_params": to_numpy_tree( single_grid_params.params ),
        "binary_params": to_numpy_tree( binary_grid_params.params ),
        "files": [ exp.filename for exp in exposures_single ],
    }


    # Save

    if save:

        safe_target = ( str(target) .replace(" ", "_") .replace("/", "_") )
        group_dir = ( Path(results_dir) / safe_target / filt )
        group_dir.mkdir( parents=True, exist_ok=True )
        save_path = ( group_dir / "grid.pkl" )

        with open( save_path, "wb" ) as f: pickle.dump( grid_output, f )
        print( "Saved:", save_path )
        grid_output[ "save_path" ] = str( save_path )


    return grid_output